# Judge Reliability — oracle ICC + second-judge agreement  `[MEASUREMENT VALIDITY]`

Buys down **LIMITATIONS.md §1** (judge reliability is not measured) and **§2** (patient = oracle coupling) on a
cheap SUBSET. Two parts, each behind an explicit `RUN_*` flag so opening this notebook never spends money:

1. **Repeatability (ICC)** — re-score the anchor models' conversations `N_REPS` times with the PRIMARY oracle
   (gpt-4o-mini, same temperature, per-rep seeds — the main pipeline pins `seed=42`, so its reps would be
   trivially identical). → per-metric **ICC(2,1)** + mean |Δ|, upgrading the "oracle noise ≈ 0.10" folklore into a citable statistic.
2. **Second judge** — score the same conversations once with a **different judge** (pluggable: OpenAI or **Claude**
   via the `anthropic` SDK). → per-metric agreement (r/ρ/bias) vs the primary oracle **and the defense-critical check**:
   does the PTO-vs-GRPO endpoint contrast survive a judge that did NOT simulate the patient?

Outputs land in `data/judge_check/` (never the real `eval_scores/`); scoring is resume-safe (existing CSVs skipped).
Summary tables are written to `data/judge_check/summary/` — surface them in `LIMITATIONS.md` / `4_Training_and_Reliability`
once real numbers exist. This notebook is a scoring pipeline like `Run_Eval.ipynb` — it is NOT part of `render_views.py`.

In [ ]:
import sys, os, asyncio
sys.path.insert(0, os.path.abspath("."))
import numpy as np, pandas as pd
pd.set_option("display.width", 185, "display.max_columns", 50)
from eda_analysis.scoring import registry as osc_config, conversations as osc_data, judge as jc

# ╔═══ KNOBS ═══════════════════════════════════════════════════════════════════╗
# Anchor model states (base + endpoints + the GRPO peak). file_index pairing across arms is
# persona-valid ONLY at matched iterations (same seed+k+1 shuffle), which these are.
SUBSET_MODELS = ["PTOExp3_LA0_Base", "PTOExp3_LA0_I10", "GRPOExp3_LA0_I8", "GRPOExp3_LA0_I10"]
METRICS   = ["Q1", "Q2", "MICI"]   # Q1+Q2 = the headline reward metric; MICI = the sycophancy claim
SUBSET_N  = None                     # None = all 96 convs per model; or an int to shrink further
N_REPS    = 3                        # ICC repetitions with the primary oracle
CONCURRENCY = 16

# The second judge — LIOR DECIDES THE MODEL. Claude options (input/output $ per MTok):
#   claude-haiku-4-5  ($1/$5, economical default) · claude-sonnet-5 ($3/$15) · claude-opus-4-8 ($5/$25)
# Requires `pip install anthropic` + a key in env ANTHROPIC_API_KEY or anthropic_key.txt at the
# experiment root (beside openai_key.txt). Provider "openai" also works (e.g. a different GPT).
SECOND_JUDGE = jc.JudgeSpec(provider="anthropic", model="claude-haiku-4-5")

# Money switches — nothing is scored until you flip these to True and run the cell below.
RUN_ICC          = False
RUN_SECOND_JUDGE = False
# ╚═════════════════════════════════════════════════════════════════════════════╝

PRIMARY = jc.PRIMARY_JUDGE
print("primary judge:", PRIMARY.tag, "| second judge:", SECOND_JUDGE.tag)
print("judge_check root:", jc.JUDGE_CHECK_ROOT)

In [ ]:
# Load the subset conversations (from the auto-discovered registry) + preview the call count/cost.
exps = [e for e in osc_config.EXPERIMENTS if e.model_name in SUBSET_MODELS]
assert {e.model_name for e in exps} == set(SUBSET_MODELS), \
    f"missing on disk: {set(SUBSET_MODELS) - {e.model_name for e in exps}} (Drive symlinks mounted?)"
paths  = osc_config.resolve_paths(exps)
names  = [e.model_name for e in exps]
layout = osc_config.get_model_eval_layout(exps)
combined = osc_data.combine_data(osc_data.load_data(paths), names)
n_convs = combined.groupby("Model").size().min() if SUBSET_N is None else SUBSET_N
print(combined.groupby("Model").size().rename("convs on disk"))

# Cost preview (rough: ~2.5k input + ~0.15k output tokens per oracle call).
calls_icc   = len(SUBSET_MODELS) * n_convs * len(METRICS) * N_REPS
calls_judge = len(SUBSET_MODELS) * n_convs * len(METRICS)
IN_TOK, OUT_TOK = 2500, 150
price = {"gpt-4o-mini": (0.15, 0.60), "claude-haiku-4-5": (1.0, 5.0),
         "claude-sonnet-5": (3.0, 15.0), "claude-opus-4-8": (5.0, 25.0)}
def est(n_calls, model_key):
    pin, pout = next(v for k, v in price.items() if k in model_key)
    return n_calls * (IN_TOK * pin + OUT_TOK * pout) / 1e6
print(f"\nICC: {calls_icc} calls with {PRIMARY.model} ≈ ${est(calls_icc, 'gpt-4o-mini'):.2f}")
try:
    print(f"2nd judge: {calls_judge} calls with {SECOND_JUDGE.model} ≈ ${est(calls_judge, SECOND_JUDGE.model):.2f}")
except StopIteration:
    print(f"2nd judge: {calls_judge} calls with {SECOND_JUDGE.model} (add its price to `price` for an estimate)")

## 1 · Repeatability — primary oracle × N_REPS  `[$ — flips on RUN_ICC]`
Rep 0 deliberately mirrors the production configuration; reps differ only in the API `seed` (temperature stays 0.1),
so the ICC captures pure sampling noise of the measurement instrument.

In [ ]:
if RUN_ICC:
    for rep in range(N_REPS):
        await jc.run_judge_scoring(PRIMARY, combined, METRICS, layout,
                                   rep=rep, concurrency=CONCURRENCY, subset_n=SUBSET_N)
else:
    print("RUN_ICC=False — skipping (flip the knob in cell 1 to score).")

In [ ]:
# ICC(2,1) + mean |Δ| between reps, per (metric, model) — runs on whatever is on disk.
icc_long = jc.load_judge_scores(PRIMARY.tag)
if icc_long.empty or icc_long.rep.nunique() < 2:
    print("Need ≥2 scored reps for ICC — nothing to report yet.")
    REP_TAB = None
else:
    REP_TAB = jc.repeatability_table(icc_long)
    display(REP_TAB)
    print("Read: ICC(2,1) ≥ 0.75 = good, ≥ 0.9 = excellent test-retest reliability (Koo & Li 2016 guidelines);",
          "mean_abs_diff is the per-conversation |Δ| between reps (compare to the ~0.10 folklore figure).")

## 2 · Second judge — decoupled from the patient simulator  `[$ — flips on RUN_SECOND_JUDGE]`
The simulated patient and the primary grader are the same model (gpt-4o-mini). A second judge from a **different
family** breaks that coupling. Two questions, in order of importance: **(a)** does the PTO-vs-GRPO endpoint contrast
survive? **(b)** how well do the per-conversation scores agree (r / ρ / bias)?

In [ ]:
if RUN_SECOND_JUDGE:
    await jc.run_judge_scoring(SECOND_JUDGE, combined, METRICS, layout,
                               rep=0, concurrency=CONCURRENCY, subset_n=SUBSET_N)
else:
    print("RUN_SECOND_JUDGE=False — skipping (flip the knob in cell 1 to score).")

In [ ]:
# Primary-oracle scores for the same (model, metric, conversation) cells, from the REAL eval_scores tree.
def load_primary_long(models, metrics):
    rows = []
    for model in models:
        entry = layout[model]
        for name in metrics:
            sub, col = jc.JUDGE_METRIC_COLS[name]
            ddir = osc_config.eval_csv_dir(entry["root"], entry["oracle"], sub, model)
            if not os.path.isdir(ddir):
                continue
            for fn in os.listdir(ddir):
                stem, ext = os.path.splitext(fn)
                if ext != ".csv" or not stem.isdigit():
                    continue
                try:
                    df = pd.read_csv(os.path.join(ddir, fn))
                except Exception:
                    continue
                if len(df) and col in df.columns:
                    rows.append({"metric": name, "model": model, "file_index": int(stem),
                                 "value": float(df[col].iloc[0])})
    return pd.DataFrame(rows)

judge_long   = jc.load_judge_scores(SECOND_JUDGE.tag, reps=[0])
primary_long = load_primary_long(SUBSET_MODELS, METRICS)
if judge_long.empty:
    print("Second judge has no scores on disk yet.")
    AGR_TAB = None; CONTRASTS = None
else:
    AGR_TAB = jc.agreement_table(judge_long, primary_long)
    display(AGR_TAB)
    # THE defense check: is the endpoint contrast judge-independent?
    CONTRASTS = pd.DataFrame([
        jc.contrast_preservation(judge_long, primary_long, "PTOExp3_LA0_I10", "GRPOExp3_LA0_I10", m)
        for m in METRICS
    ] + [
        jc.contrast_preservation(judge_long, primary_long, "PTOExp3_LA0_I10", "PTOExp3_LA0_Base", m)
        for m in METRICS
    ])
    display(CONTRASTS)
    print("Read: `same_sign=True` on the PTO−GRPO endpoint rows = the headline result is not an artifact",
          "of the shared patient/oracle model. For MICI remember lower = better (a positive Δ is worse).")

In [ ]:
# Persist whatever exists to data/judge_check/summary/ (md + csv) — cite these in LIMITATIONS.md.
SUMMARY_DIR = os.path.join(jc.JUDGE_CHECK_ROOT, "summary")
os.makedirs(SUMMARY_DIR, exist_ok=True)
def save(df, name):
    if df is None or (hasattr(df, "empty") and df.empty):
        return
    df.to_csv(os.path.join(SUMMARY_DIR, f"{name}.csv"), index=False)
    with open(os.path.join(SUMMARY_DIR, f"{name}.md"), "w", encoding="utf-8") as f:
        f.write(df.to_markdown(index=False))
    print("saved", name)
save(globals().get("REP_TAB"), "oracle_repeatability_icc")
save(globals().get("AGR_TAB"), f"second_judge_agreement_{SECOND_JUDGE.tag}")
save(globals().get("CONTRASTS"), f"second_judge_contrasts_{SECOND_JUDGE.tag}")

## How to read / next steps
- **ICC ≥ 0.75** on Q1/Q2 → the per-conversation scores are reliable; the paired 96-persona design then makes the
  arm-level contrasts far more reliable still (means over 96 convs shrink noise ~√96).
- **Contrast preservation** is the load-bearing result for the thesis: if the second judge (different model family,
  never saw the patient role) reproduces the sign of PTO−GRPO at iter 10 and the vs-base gains, LIMITATIONS §2's
  coupling concern is empirically bounded, not just acknowledged.
- Agreement (r/ρ) will be attenuated by both judges' own noise — interpret against the ICC ceiling
  (max expected r ≈ √(ICC_primary × ICC_judge)), not against 1.0.
- After running: update **LIMITATIONS.md §1/§2** with the measured numbers + point to `data/judge_check/summary/`.
- Cost levers: `SUBSET_N`, fewer `METRICS`, fewer `SUBSET_MODELS`. The defaults (4 models × 96 convs × 3 metrics)
  are a few dollars on gpt-4o-mini/haiku — see the preview cell.